# Telco Customer Churn Analysis & Prediction

## Project goal
This project explores which customer characteristics are associated with churn and builds baseline classification models to predict whether a customer will leave.

**Business questions**
1. Which customer groups have higher churn rates?
2. How well can we predict churn using available customer information?
3. What should the business investigate further?

The analysis identifies associations, not proven causes.

## 1. Load and inspect the data

We first check the dataset dimensions, data types, missing values, and target distribution. This helps us understand data quality before analysis.

In [10]:

import os
import pandas as pd
from IPython.display import display

# Ищем CSV-файлы, подключённые к текущему Kaggle Notebook.
# Это надёжнее, чем угадывать путь к конкретному датасету.
csv_files = []

for root, dirs, files in os.walk("/kaggle/input"):
    for filename in files:
        if filename.lower().endswith(".csv"):
            csv_files.append(os.path.join(root, filename))

# Выводим найденные файлы, чтобы проверить доступность датасета.
print("Найденные CSV-файлы:")

for path in csv_files:
    print(path)

# Ищем именно датасет Telco Customer Churn.
matches = [
    path for path in csv_files
    if "WA_Fn-UseC_-Telco-Customer-Churn.csv".lower()
    == os.path.basename(path).lower()
]

if len(matches) == 0:
    raise FileNotFoundError(
        "Датасет не найден. Подключи Telco Customer Churn "
        "через Add Input в Kaggle Notebook."
    )

# Загружаем найденный файл.
df = pd.read_csv(matches[0])

# Преобразуем TotalCharges в число; пустые строки станут NaN.
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"], errors="coerce"
)

print("\nРазмер датасета:", df.shape)
display(df.head())


Найденные CSV-файлы:
/kaggle/input/datasets/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv

Размер датасета: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [11]:
import pandas as pd
import numpy as np

from IPython.display import display

path = "/kaggle/input/datasets/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = pd.read_csv(path)

print("Dataset shape:", df.shape)
display(df.head())

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nMissing values:")
display(df.isna().sum().to_frame("missing_values"))

print("\nChurn distribution:")
display(df["Churn"].value_counts().to_frame("customers"))
display((df["Churn"].value_counts(normalize=True) * 100).round(2).to_frame("percent"))

Dataset shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes



Data types:


,dtype
customerID,object
gender,object
SeniorCitizen,int64
Partner,object
Dependents,object
tenure,int64
PhoneService,object
MultipleLines,object
InternetService,object
OnlineSecurity,object



Missing values:


,missing_values
customerID,0
gender,0
SeniorCitizen,0
Partner,0
Dependents,0
tenure,0
PhoneService,0
MultipleLines,0
InternetService,0
OnlineSecurity,0



Churn distribution:


,customers
Churn,
No,5174
Yes,1869


,percent
Churn,
No,73.46
Yes,26.54


## 2. Clean `TotalCharges`

`TotalCharges` contains blank strings for some customers with zero tenure. We convert the column to numeric; blank or invalid values become missing values and will be imputed inside the model pipeline.

Keeping imputation inside the pipeline prevents the preprocessing rules from being learned using validation data.

In [12]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

print("Missing TotalCharges after conversion:", df["TotalCharges"].isna().sum())
print("Customers with zero tenure:", (df["tenure"] == 0).sum())

Missing TotalCharges after conversion: 11
Customers with zero tenure: 11


## 3. Exploratory data analysis

We compare churn rates across a few business-relevant variables. These are descriptive comparisons: they show where churn is more common, but do not prove why customers leave.

In [13]:
# Churn rate by contract type
contract_churn = pd.crosstab(
    df["Contract"], df["Churn"], normalize="index"
) * 100

print("Churn distribution by contract (%):")
display(contract_churn.round(2))

# Tenure and monthly charges by churn status
print("Tenure by churn status:")
display(df.groupby("Churn")["tenure"].agg(["count", "mean", "median"]).round(2))

print("Monthly charges by churn status:")
display(df.groupby("Churn")["MonthlyCharges"].agg(["mean", "median"]).round(2))

# Churn rate by internet service
internet_churn = pd.crosstab(
    df["InternetService"], df["Churn"], normalize="index"
) * 100

print("Churn distribution by internet service (%):")
display(internet_churn.round(2))

# Churn rate by Online Security
security_churn = pd.crosstab(
    df["OnlineSecurity"], df["Churn"], normalize="index"
) * 100

print("Churn distribution by Online Security (%):")
display(security_churn.round(2))

Churn distribution by contract (%):


Churn,No,Yes
Contract,,
Month-to-month,57.29,42.71
One year,88.73,11.27
Two year,97.17,2.83


Tenure by churn status:


,count,mean,median
Churn,,,
No,5174,37.57,38.0
Yes,1869,17.98,10.0


Monthly charges by churn status:


,mean,median
Churn,,
No,61.27,64.43
Yes,74.44,79.65


Churn distribution by internet service (%):


Churn,No,Yes
InternetService,,
DSL,81.04,18.96
Fiber optic,58.11,41.89
No,92.60,7.40


Churn distribution by Online Security (%):


Churn,No,Yes
OnlineSecurity,,
No,58.23,41.77
No internet service,92.60,7.40
Yes,85.39,14.61


### EDA summary

In this dataset, churn is higher among month-to-month customers and fiber-optic internet customers. Customers who churned also tend to have shorter tenure.

These patterns help prioritize further investigation. They should not be interpreted as causal effects.

## 4. Prepare data for modeling

The target is encoded as `1` for churn and `0` for no churn. We exclude `customerID` because it is an identifier, not a useful customer behavior feature.

We use a stratified 80/20 split so that the churn proportion is similar in the training and validation sets.

In [14]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

X = df.drop(columns=["customerID", "Churn"])
y = (df["Churn"] == "Yes").astype(int)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numeric_cols = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = X_train.select_dtypes(include=["object"]).columns.tolist()

# Numeric columns: median imputation and scaling.
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical columns: most-frequent imputation and one-hot encoding.
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

print("Train shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Overall churn rate:", round(y.mean() * 100, 2), "%")

Train shape: (5634, 19)
Validation shape: (1409, 19)
Overall churn rate: 26.54 %


## 5. Train and compare baseline models

We compare Logistic Regression, Decision Tree, and Random Forest on the same split and with the same preprocessing approach.

- **Precision:** among customers predicted to churn, how many actually churned?
- **Recall:** among all customers who actually churned, how many did the model identify?
- **F1:** harmonic mean of Precision and Recall.
- **ROC-AUC:** how well the model ranks churned customers above non-churned customers across thresholds.

The 0.5 classification threshold is used for Precision, Recall, and F1 by default.

In [15]:
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score
)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, max_depth=10, random_state=42, n_jobs=-1
    )
}

fitted_models = {}
rows = []

for name, classifier in models.items():
    pipeline = Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("classifier", classifier)
    ])
    pipeline.fit(X_train, y_train)

    predictions = pipeline.predict(X_val)
    probabilities = pipeline.predict_proba(X_val)[:, 1]
    fitted_models[name] = pipeline

    rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_val, predictions),
        "Precision": precision_score(y_val, predictions, zero_division=0),
        "Recall": recall_score(y_val, predictions, zero_division=0),
        "F1": f1_score(y_val, predictions, zero_division=0),
        "ROC-AUC": roc_auc_score(y_val, probabilities)
    })

model_results = pd.DataFrame(rows).sort_values("ROC-AUC", ascending=False)
display(model_results.round(3))

,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,Logistic Regression,0.806,0.657,0.559,0.604,0.842
2,Random Forest,0.803,0.663,0.521,0.584,0.841
1,Decision Tree,0.798,0.635,0.567,0.599,0.830


## 6. Check Random Forest overfitting

We compare performance on training and validation data. A large gap can indicate overfitting: the model fits training data better than data it has not seen.

In [16]:
rf_model = fitted_models["Random Forest"]

train_pred = rf_model.predict(X_train)
val_pred = rf_model.predict(X_val)
train_prob = rf_model.predict_proba(X_train)[:, 1]
val_prob = rf_model.predict_proba(X_val)[:, 1]

overfitting_check = pd.DataFrame({
    "Dataset": ["Train", "Validation"],
    "Accuracy": [
        accuracy_score(y_train, train_pred),
        accuracy_score(y_val, val_pred)
    ],
    "F1": [
        f1_score(y_train, train_pred),
        f1_score(y_val, val_pred)
    ],
    "ROC-AUC": [
        roc_auc_score(y_train, train_prob),
        roc_auc_score(y_val, val_prob)
    ]
})

display(overfitting_check.round(3))

,Dataset,Accuracy,F1,ROC-AUC
0,Train,0.873,0.741,0.953
1,Validation,0.803,0.584,0.841


## 7. Interpret Logistic Regression coefficients

We inspect the positive coefficients and odds ratios to identify associations learned by the model.

- A positive coefficient corresponds to higher modeled log-odds of churn, holding other included features constant.
- An odds ratio above 1 indicates higher modeled odds; below 1 indicates lower modeled odds.
- For scaled numeric features, the odds ratio corresponds to a one-standard-deviation increase.
- These values are not causal effects, and odds ratios are not percentage-point changes in probability.

In [17]:
log_model = fitted_models["Logistic Regression"]

feature_names = log_model.named_steps["preprocessor"].get_feature_names_out()
coefficients = log_model.named_steps["classifier"].coef_[0]

feature_analysis = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients,
    "Odds_Ratio": np.exp(coefficients)
})

print("Features with the highest positive coefficients:")
display(
    feature_analysis.sort_values("Coefficient", ascending=False)
    .head(10).round(3)
)

Features with the highest positive coefficients:


,Feature,Coefficient,Odds_Ratio
16,cat__InternetService_Fiber optic,0.640,1.897
36,cat__Contract_Month-to-month,0.579,1.785
3,num__TotalCharges,0.517,1.676
35,cat__StreamingMovies_Yes,0.204,1.226
32,cat__StreamingTV_Yes,0.204,1.226
43,cat__PaymentMethod_Electronic check,0.198,1.219
18,cat__OnlineSecurity_No,0.157,1.170
27,cat__TechSupport_No,0.132,1.141
14,cat__MultipleLines_Yes,0.106,1.112
0,num__SeniorCitizen,0.054,1.055


## 8. Business implications and limitations

### Findings to investigate
- **Month-to-month contracts:** observed churn is substantially higher; investigate whether suitable longer-term offers could improve retention.
- **Fiber-optic customers:** investigate pricing, service quality, expectations, and support experience.
- **Additional services:** investigate whether relevant security and technical support offers meet customer needs.
- **Early customer lifecycle:** the shorter median tenure among churned customers suggests reviewing onboarding and early customer experience.

These are hypotheses for further investigation, not proven causal explanations.

### Limitations
- The dataset is observational, so associations do not establish causality.
- Model performance may vary across data splits.
- The validation metrics do not guarantee future performance.
- The current classification threshold has not been optimized using retention costs and business outcomes.

## Conclusion

This project covers exploratory analysis, preprocessing, baseline model comparison, an overfitting check, and interpretation of model coefficients. Logistic Regression provides a useful baseline; the final model choice should depend on robust validation and business costs, not a single metric alone.